# BioMechAI — Experiment v6: PoseC3D Limb Model (Threshold 80%)
### Final Year Project II (Semester 8) — Action Recognition Fine-Tuning

**Baseline to Beat (Champion Model v5)**:
- **Top-1 Accuracy**: `53.38%` (237/444 clips on frozen 115 held-out videos zero-leakage split)
- **Macro Recall**: `53.15%`
- **Top-5 Accuracy**: `91.22%`
- **Champion Weights**: `models/posec3d_v5_limb/best_acc_top1_epoch_10.pth` (8.33 MB)

**Threshold 80% Parameters Configured in this Notebook**:
1. **Landmark Proposal / Matching Threshold**: `box_thr = 0.8` (80% confidence filtering in `PoseDataset`)
2. **Gradient Clipping Threshold**: `clip_grad = dict(max_norm=80, norm_type=2)` (stabilizes backprop at scale 80)
3. **Training Epochs Budget Threshold**: `max_epochs = 80` (or set to 24 for fast ~80 min run)

**Dual Storage & Checkpoint Protection**:
- **Layer 1 (Guaranteed)**: Automatic `.zip` packaging of ALL checkpoints & logs in `/kaggle/working/` for direct 1-click Kaggle download
- **Layer 2 (Live Cloud Sync)**: Real-time Google Drive sync hook targeting folder `1XtVOZDF_4RiG2uyZ2PbmKkKPTVgW8yDK` (verified at pre-flight)

---

## ═══ CELL 1: Install Dependencies & Apply Compatibility Patches ═══
*Run this cell first in a fresh Kaggle GPU session. Installs `google-api-python-client`, `mmengine`, `mmcv` (with smart fallback), `timm`, `einops`, clones MMAction2, and applies all compatibility patches.*

In [ ]:
# ======================================================================
# CELL 1: BioMechAI Fresh Session Setup & Patches (Bypasses setup.py)
# ======================================================================
import os, sys, subprocess, functools

print("⏳ Step 1/5: Installing Google Drive client libraries...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", 
    "google-api-python-client", "google-auth-oauthlib", "google-auth-httplib2"], check=True)

print("⏳ Step 2/5: Installing mmengine...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mmengine==0.10.7"], check=True)

print("⏳ Step 3/5: Installing mmcv (with smart prebuilt wheel fallback)...")
import torch
print(f"   Detected Environment: PyTorch {torch.__version__}, CUDA {torch.version.cuda}")

installed_mmcv = False
# Attempt A: OpenMMLab prebuilt wheels for torch2.4
res = subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "mmcv==2.2.0",
    "-f", "https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html"
])
if res.returncode == 0:
    installed_mmcv = True
    print("   ✅ Installed prebuilt mmcv 2.2.0 (cu121/torch2.4)")

# Attempt B: OpenMMLab prebuilt wheels for torch2.1
if not installed_mmcv:
    res = subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "mmcv==2.2.0",
        "-f", "https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html"
    ])
    if res.returncode == 0:
        installed_mmcv = True
        print("   ✅ Installed prebuilt mmcv 2.2.0 (cu121/torch2.1)")

# Attempt C: Universal mmcv-lite (100% prebuilt on PyPI, zero build errors, full PoseC3D transforms)
if not installed_mmcv:
    print("   ℹ️ Falling back to universal mmcv-lite 2.2.0 (avoids C++ compilation mismatch)...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "mmcv-lite==2.2.0"
    ], check=True)
    installed_mmcv = True
    print("   ✅ Installed mmcv-lite 2.2.0 successfully!")

print("⏳ Step 4/5: Installing supporting packages (einops, timm)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "timm"], check=True)

print("⏳ Step 5/5: Cloning MMAction2 Framework from GitHub...")
subprocess.run(["rm", "-rf", "/kaggle/working/mmaction2"], check=False)
subprocess.run([
    "git", "clone", "--depth=1",
    "https://github.com/open-mmlab/mmaction2.git",
    "/kaggle/working/mmaction2"
], check=True)

# Apply Patches to avoid setup.py compilation crashes
print("🛠 Applying compatibility patches...")

# Patch 1: MMCV version ceiling guard (allows mmcv 2.2.0 to work)
init_path = "/kaggle/working/mmaction2/mmaction/__init__.py"
if os.path.exists(init_path):
    with open(init_path) as f:
        content = f.read()
    new_content = content.replace("mmcv_maximum_version = '2.2.0'", "mmcv_maximum_version = '2.3.0'")
    with open(init_path, "w") as f:
        f.write(new_content)
    print("✅ Patch 1: MMCV ceiling guard set to 2.3.0")

# Patch 2: Disable unused multimodal module (avoids PIL/HuggingFace conflicts)
mm_init = "/kaggle/working/mmaction2/mmaction/models/multimodal/__init__.py"
if os.path.exists(os.path.dirname(mm_init)):
    os.makedirs(os.path.dirname(mm_init), exist_ok=True)
    with open(mm_init, "w") as f:
        f.write("__all__ = []\n")
    print("✅ Patch 2: Multimodal module disabled")

# Patch 3: np.Inf -> np.inf (NumPy 2.0 compatibility)
patched_files = 0
for root, dirs, files in os.walk("/kaggle/working/mmaction2"):
    for fname in files:
        if fname.endswith(".py"):
            fpath = os.path.join(root, fname)
            try:
                with open(fpath) as f:
                    txt = f.read()
                if "np.Inf" in txt:
                    with open(fpath, "w") as f:
                        f.write(txt.replace("np.Inf", "np.inf"))
                    patched_files += 1
            except Exception:
                pass
print(f"✅ Patch 3: np.Inf patched in {patched_files} files")

# Patch 4: PyTorch 2.6+ weights_only & C++ ABI mismatch in tools/train.py
train_py = "/kaggle/working/mmaction2/tools/train.py"
if os.path.exists(train_py):
    with open(train_py) as f:
        train_content = f.read()
    patch_header = """import functools
import torch as _torch
_orig_torch_load = _torch.load
_torch.load = functools.partial(_orig_torch_load, weights_only=False)
try:
    import mmengine.model.utils as _mu
    _mu.mmcv_full_available = lambda: False
except Exception:
    pass
"""
    if "functools" not in train_content:
        with open(train_py, "w") as f:
            f.write(patch_header + train_content)
        print("✅ Patch 4: tools/train.py patched")

# Add mmaction2 directly to sys.path so no pip install -e is required
if "/kaggle/working/mmaction2" not in sys.path:
    sys.path.insert(0, "/kaggle/working/mmaction2")
os.environ["PYTHONPATH"] = "/kaggle/working/mmaction2:" + os.environ.get("PYTHONPATH", "")

print("\n🎉 Cell 1 Complete: Environment ready for training!")


## ═══ CELL 2: Pre-Flight Check, Drive Sync Hook, Transforms & Training ═══
*This cell:
1. Performs a live Pre-Flight Test of GPU, Datasets, and Google Drive access
2. Generates `pose_transforms_extra.py` (RandomRotateKeypoints) & `drive_sync_hook.py`
3. Generates the PoseC3D configuration with **Threshold = 80%** (`box_thr = 0.8`, `max_norm = 80`, `max_epochs = 80`)
4. Runs training, saves checkpoints in real-time, and guarantees `.zip` package generation in `/kaggle/working/`*

In [ ]:
# ======================================================================
# CELL 2: Pre-Flight Check, Transforms, Threshold 80% Config & Training
# ======================================================================
import os, sys, glob, shutil, subprocess, zipfile, torch

# ── TARGET GOOGLE DRIVE FOLDER ─────────────────────────────────────────
DRIVE_FOLDER_ID = "1XtVOZDF_4RiG2uyZ2PbmKkKPTVgW8yDK"

# ── 1. Hardware Check ──────────────────────────────────────────────────
print("=" * 65)
print("1. HARDWARE CHECK")
print("=" * 65)
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ No GPU detected! Please change Kaggle Accelerator to GPU T4 x2 or P100.")

# ── 2. Dataset Discovery ───────────────────────────────────────────────
print("\n" + "=" * 65)
print("2. DATASET VERIFICATION")
print("=" * 65)
for fn in ["custom_dataset_train.pkl", "custom_dataset_val.pkl", "token.json"]:
    dst = f"/kaggle/working/{fn}"
    if os.path.exists(dst):
        print(f"✅ {fn} already in /kaggle/working/ ({os.path.getsize(dst)/1024**2:.2f} MB)")
        continue
    found = glob.glob(f"/kaggle/input/**/{fn}", recursive=True)
    if found:
        shutil.copy(found[0], dst)
        print(f"✅ Auto-copied {fn} -> /kaggle/working/ ({os.path.getsize(dst)/1024**2:.2f} MB)")
    else:
        if fn == "token.json":
            print("ℹ️ token.json not found in /kaggle/input (Drive upload will be disabled; local zip is guaranteed).")
        else:
            print(f"❌ ERROR: {fn} NOT FOUND in /kaggle/input! Attach your dataset.")

TRAIN_PKL = "/kaggle/working/custom_dataset_train.pkl"
VAL_PKL   = "/kaggle/working/custom_dataset_val.pkl"
TOKEN_JSON= "/kaggle/working/token.json"
WORK_DIR  = "/kaggle/working/work_dirs/posec3d_v6_thr80"
CONFIG_PY = "/kaggle/working/posec3d_biomechai_v6_thr80.py"
ZIP_PATH  = "/kaggle/working/BioMechAI_PoseC3D_v6_thr80_artifacts.zip"
os.makedirs(WORK_DIR, exist_ok=True)

# ── 3. Google Drive Live Pre-Flight Connection Test ────────────────────
print("\n" + "=" * 65)
print("3. GOOGLE DRIVE PRE-FLIGHT TEST")
print("=" * 65)
drive_active = False
if os.path.exists(TOKEN_JSON):
    try:
        from google.oauth2.credentials import Credentials
        from google.auth.transport.requests import Request
        from googleapiclient.discovery import build
        creds = Credentials.from_authorized_user_file(
            TOKEN_JSON, scopes=['https://www.googleapis.com/auth/drive'])
        if creds.expired and creds.refresh_token:
            creds.refresh(Request())
        drive_svc = build('drive', 'v3', credentials=creds)
        folder_info = drive_svc.files().get(fileId=DRIVE_FOLDER_ID, fields='id, name').execute()
        print(f"✅ Google Drive Connected! Target folder: '{folder_info.get('name')}' (ID: {DRIVE_FOLDER_ID})")
        print("   Checkpoints will automatically sync to Google Drive after every 2 epochs.")
        drive_active = True
    except Exception as drive_err:
        print(f"⚠️ Google Drive Token Notice: {drive_err}")
        print("   * Safety Protection: Training will proceed safely using Kaggle's local storage.")
        print("   * All checkpoints will be automatically saved and zipped to:")
        print(f"     {ZIP_PATH} (Downloadable with 1 click from Kaggle Output tab).")
else:
    print("ℹ️ No token.json provided. Checkpoints will be saved locally & zipped.")

# ── 4. Create Custom Augmentation (RandomRotateKeypoints) & Drive Hook ──
print("\n" + "=" * 65)
print("4. REGISTERING CUSTOM EXTENSIONS")
print("=" * 65)

# Write RandomRotateKeypoints transform
rotate_transform_code = r'''import numpy as np
from mmcv.transforms import BaseTransform
from mmaction.registry import TRANSFORMS

@TRANSFORMS.register_module()
class RandomRotateKeypoints(BaseTransform):
    def __init__(self, max_angle=12.0, prob=0.5):
        self.max_angle = max_angle
        self.prob = prob
    def transform(self, results):
        if np.random.rand() > self.prob:
            return results
        angle = np.random.uniform(-self.max_angle, self.max_angle)
        rad = np.deg2rad(angle)
        cos_a, sin_a = np.cos(rad), np.sin(rad)
        h, w = results['img_shape'][:2]
        cx, cy = w / 2.0, h / 2.0
        kps = results['keypoint']
        x = kps[..., 0] - cx
        y = kps[..., 1] - cy
        new_x = cos_a * x - sin_a * y + cx
        new_y = sin_a * x + cos_a * y + cy
        new_x = np.clip(new_x, 0, w - 1)
        new_y = np.clip(new_y, 0, h - 1)
        results['keypoint'] = np.stack([new_x, new_y], axis=-1).astype(kps.dtype)
        return results
'''
with open('/kaggle/working/pose_transforms_extra.py', 'w') as f:
    f.write(rotate_transform_code)
print("✅ RandomRotateKeypoints registered (/kaggle/working/pose_transforms_extra.py)")

# Write Google Drive Hook (if active)
if drive_active:
    hook_code = r'''import os
from mmengine.hooks import Hook
from mmengine.registry import HOOKS
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

@HOOKS.register_module()
class DriveCheckpointSyncHook(Hook):
    priority = 'LOWEST'
    def __init__(self, token_json_path, drive_folder_id, work_dir):
        self.token_json_path = token_json_path
        self.drive_folder_id = drive_folder_id
        self.work_dir = work_dir
        self._uploaded = set()
    def _get_service(self):
        creds = Credentials.from_authorized_user_file(
            self.token_json_path, scopes=['https://www.googleapis.com/auth/drive'])
        if creds.expired and creds.refresh_token:
            creds.refresh(Request())
        return build('drive', 'v3', credentials=creds)
    def after_train_epoch(self, runner):
        if not os.path.isdir(self.work_dir):
            return
        try:
            service = self._get_service()
            for fname in sorted(os.listdir(self.work_dir)):
                if fname.endswith('.pth'):
                    fpath = os.path.join(self.work_dir, fname)
                    if fpath not in self._uploaded:
                        runner.logger.info(f'Uploading {fname} to Google Drive...')
                        meta = {'name': fname, 'parents': [self.drive_folder_id]}
                        media = MediaFileUpload(fpath, resumable=True)
                        res = service.files().create(body=meta, media_body=media, fields='id').execute()
                        self._uploaded.add(fpath)
                        runner.logger.info(f'Drive Backup: {fname} synced! ID: {res.get("id")}')
        except Exception as e:
            runner.logger.warning(f'Drive sync exception: {e}')
'''
    with open("/kaggle/working/drive_sync_hook.py", "w") as f:
        f.write(hook_code)
    print("✅ DriveCheckpointSyncHook registered (/kaggle/working/drive_sync_hook.py)")

# ── 5. Generate PoseC3D Configuration with Threshold 80% ──────────────
print("\n" + "=" * 65)
print("5. GENERATING CONFIGURATION (THRESHOLD 80%)")
print("=" * 65)

imports_list = ['pose_transforms_extra']
if drive_active:
    imports_list.append('drive_sync_hook')
custom_imports_str = f"custom_imports = dict(imports={imports_list}, allow_failed_imports=True)"

custom_hooks_str = f"""custom_hooks = [
    dict(type='DriveCheckpointSyncHook',
         token_json_path='{TOKEN_JSON}',
         drive_folder_id='{DRIVE_FOLDER_ID}',
         work_dir='{WORK_DIR}')
]""" if drive_active else "custom_hooks = []"

config_content = f"""default_scope = 'mmaction'
{custom_imports_str}

# Official OpenMMLab FineGYM Pretrained SlowOnly PoseC3D Limb Checkpoint
load_from = 'https://download.openmmlab.com/mmaction/v1.0/skeleton/posec3d/slowonly_r50_8xb16-u48-240e_gym-limb/slowonly_r50_8xb16-u48-240e_gym-limb_20220815-2e6e3c5c.pth'

model = dict(
    type='Recognizer3D',
    backbone=dict(
        type='ResNet3dSlowOnly',
        depth=50,
        pretrained=None,
        in_channels=17,
        base_channels=32,
        num_stages=3,
        out_indices=(2,),
        stage_blocks=(4, 6, 3),
        conv1_stride_s=1,
        pool1_stride_s=1,
        inflate=(0, 1, 1),
        spatial_strides=(2, 2, 2),
        temporal_strides=(1, 1, 2),
        dilations=(1, 1, 1)),
    cls_head=dict(
        type='I3DHead',
        in_channels=512,
        num_classes=7,
        dropout_ratio=0.6,
        average_clips='prob'))

dataset_type = 'PoseDataset'
data_root = './'
ann_file_train = '{TRAIN_PKL}'
ann_file_val = '{VAL_PKL}'

left_kp = [1, 3, 5, 7, 9, 11, 13, 15]
right_kp = [2, 4, 6, 8, 10, 12, 14, 16]
skeletons = [
    [0, 5], [0, 6], [5, 7], [7, 9], [6, 8], [8, 10], [5, 11],
    [11, 13], [13, 15], [6, 12], [12, 14], [14, 16], [0, 1], [0, 2],
    [1, 3], [2, 4], [11, 12]
]

train_pipeline = [
    dict(type='UniformSampleFrames', clip_len=48),
    dict(type='PoseDecode'),
    dict(type='PoseCompact', hw_ratio=1., allow_imgpad=True),
    dict(type='Resize', scale=(-1, 64)),
    dict(type='RandomResizedCrop', area_range=(0.56, 1.0)),
    dict(type='Resize', scale=(56, 56), keep_ratio=False),
    dict(type='Flip', flip_ratio=0.5, left_kp=left_kp, right_kp=right_kp),
    dict(type='RandomRotateKeypoints', max_angle=12.0, prob=0.5),
    dict(type='GeneratePoseTarget', sigma=0.6, use_score=True, with_kp=False, with_limb=True, skeletons=skeletons),
    dict(type='FormatShape', input_format='NCTHW_Heatmap'),
    dict(type='PackActionInputs')
]

val_pipeline = [
    dict(type='UniformSampleFrames', clip_len=48, num_clips=1, test_mode=True),
    dict(type='PoseDecode'),
    dict(type='PoseCompact', hw_ratio=1., allow_imgpad=True),
    dict(type='Resize', scale=(-1, 56)),
    dict(type='CenterCrop', crop_size=56),
    dict(type='GeneratePoseTarget', sigma=0.6, use_score=True, with_kp=False, with_limb=True, skeletons=skeletons),
    dict(type='FormatShape', input_format='NCTHW_Heatmap'),
    dict(type='PackActionInputs')
]

train_dataloader = dict(
    batch_size=16,
    num_workers=2,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=True),
    dataset=dict(
        type=dataset_type,
        ann_file=ann_file_train,
        pipeline=train_pipeline,
        box_thr=0.8))  # <-- 80% Landmark Proposal/Matching Threshold

val_dataloader = dict(
    batch_size=16,
    num_workers=2,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type=dataset_type,
        ann_file=ann_file_val,
        pipeline=val_pipeline,
        test_mode=True,
        box_thr=0.8))  # <-- 80% Landmark Proposal/Matching Threshold

test_dataloader = dict(
    batch_size=16,
    num_workers=2,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type=dataset_type,
        ann_file=ann_file_val,
        pipeline=val_pipeline,
        test_mode=True,
        box_thr=0.8))

# Optimizer with Gradient Clipping Threshold = 80
optim_wrapper = dict(
    optimizer=dict(type='SGD', lr=0.01, momentum=0.9, weight_decay=0.0005),
    clip_grad=dict(max_norm=80, norm_type=2))  # <-- Gradient norm threshold 80

# Training schedule with max_epochs = 80 (or set to 24 for fast ~75 min run)
train_cfg = dict(type='EpochBasedTrainLoop', max_epochs=80, val_interval=2)
val_cfg = dict(type='ValLoop')
val_evaluator = [dict(type='AccMetric')]
test_cfg = dict(type='TestLoop')
test_evaluator = [dict(type='AccMetric')]

default_hooks = dict(
    checkpoint=dict(type='CheckpointHook', interval=2, max_keep_ckpts=5, save_best='acc/top1'),
    logger=dict(type='LoggerHook', interval=20))

{custom_hooks_str}
"""

with open(CONFIG_PY, "w") as f:
    f.write(config_content.strip())
print(f"✅ Config generated at {CONFIG_PY}")
print("   • Landmark Threshold:  box_thr = 0.8 (80%)")
print("   • Gradient Clip:       max_norm = 80")
print("   • Epoch Budget:        max_epochs = 80")

# ── 6. Launch Training ────────────────────────────────────────────────
print("\n" + "=" * 65)
print("6. LAUNCHING POSEC3D TRAINING")
print("=" * 65)
cmd = [
    sys.executable, "/kaggle/working/mmaction2/tools/train.py",
    CONFIG_PY,
    "--work-dir", WORK_DIR
]

env = os.environ.copy()
env["PYTHONPATH"] = "/kaggle/working:/kaggle/working/mmaction2:" + env.get("PYTHONPATH", "")

train_res = subprocess.run(cmd, env=env)

if train_res.returncode != 0:
    print(f"❌ Training exited with error code {train_res.returncode}.")
else:
    print("\n🎉 TRAINING COMPLETE!")
    
# ── 7. Packaging Artifacts (.ZIP) ─────────────────────────────────────
print("\n" + "=" * 65)
print("7. PACKAGING ARTIFACTS (.ZIP)")
print("=" * 65)
ckpts = glob.glob(f"{WORK_DIR}/*.pth")
print(f"Found {len(ckpts)} checkpoint file(s) in {WORK_DIR}:")
for c in ckpts:
    print(f"   - {os.path.basename(c)} ({os.path.getsize(c)/1024**2:.2f} MB)")

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(CONFIG_PY, arcname=os.path.basename(CONFIG_PY))
    for c in ckpts:
        zipf.write(c, arcname=f"checkpoints/{os.path.basename(c)}")
    for log_f in glob.glob(f"{WORK_DIR}/*.json") + glob.glob(f"{WORK_DIR}/*.log"):
        zipf.write(log_f, arcname=f"logs/{os.path.basename(log_f)}")

print("\n" + "=" * 65)
print(f"📦 ZIP ARCHIVE READY: {ZIP_PATH}")
print(f"   Archive Size: {os.path.getsize(ZIP_PATH)/1024**2:.2f} MB")
print("   ✅ This zip is permanently saved in Kaggle Output tab!")
print("   ✅ Click the 3 dots next to it in Kaggle Output -> 'Download'!")
print("=" * 65)